In [ ]:
# Install required packages (run once)
!pip install -q qiskit qiskit-ibm-runtime matplotlib

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import plot_histogram, plot_distribution

# --------------------------------------------------------------------
# 1️⃣  Set up IBM Quantum service – provide your API token
# --------------------------------------------------------------------
# You can store your IBM Quantum token in an environment variable called
# IBM_QUANTUM_TOKEN or paste it directly as a string below.
# Example: token = 'YOUR_IBM_QUANTUM_TOKEN'
token = os.getenv('IBM_QUANTUM_TOKEN')
if not token:
    raise RuntimeError('IBM Quantum token not found. Set IBM_QUANTUM_TOKEN env var.')

service = QiskitRuntimeService(token=token)
# Choose the least‑busy real device (simulator=False)
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=1)
print(f'🚀 Running on real QPU: {backend.name} ({backend.num_qubits} qubits)')

# --------------------------------------------------------------------
# 2️⃣  Build a single‑qubit circuit
# --------------------------------------------------------------------
circuit = QuantumCircuit(1, 1)
circuit.h(0)   # Hadamard – superposition
circuit.t(0)   # T‑gate – phase shift
circuit.x(0)   # X‑gate – NOT
circuit.y(0)   # Y‑gate
circuit.z(0)   # Z‑gate
circuit.measure(0, 0)

# Visualise the circuit (Matplotlib backend)
circuit.draw('mpl')  # display in the notebook

# --------------------------------------------------------------------
# 3️⃣  Transpile for the target backend
# --------------------------------------------------------------------
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(circuit)

# --------------------------------------------------------------------
# 4️⃣  Execute on the real device (1024 shots)
# --------------------------------------------------------------------
sampler = Sampler(backend)
job = sampler.run([isa_circuit], shots=1024)
print(f'
⏳ Job submitted, ID: {job.job_id()}')
print('Waiting for result…')
result = job.result()
pub_result = result[0]
counts = pub_result.data.c.get_counts()
print('
✅ Job finished. Raw counts:', counts)

# --------------------------------------------------------------------
# 5️⃣  Analytics & visualisation
# --------------------------------------------------------------------
total_shots = sum(counts.values())
print('
' + '='*40)
print('📊 Statistical report')
print('='*40)
print(f'Total shots: {total_shots}')
for bitstring, cnt in sorted(counts.items()):
    pct = (cnt/total_shots)*100
    print(f' |{bitstring}⟩ : {cnt} ({pct:.2f}%)')

# Histogram (Qiskit)
plt.figure(figsize=(8,5))
plot_histogram(counts, title=f'Results on {backend.name}')
plt.show()

# Distribution (probability)
plt.figure(figsize=(8,5))
plot_distribution(counts, title='Probability distribution')
plt.show()

# Pie chart
states = list(counts.keys())
values = list(counts.values())
plt.figure(figsize=(6,6))
plt.pie(values, labels=[f'|{s}⟩' for s in states], autopct='%1.1f%%', startangle=140)
plt.title(f'Outcome fractions on {backend.name}')
plt.show()